# Transformer and BERT Chapter 17 Workshop 4

##  Download packaging

In [1]:
!pip install ktrain

In [3]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf

## Load module

In [4]:
import pandas as pd
import numpy as np

import ktrain
from ktrain import text

## Load dataset

In [5]:
import os
# from google.colab import drive
# drive.mount('/content/drive')
print(os.path.exists("./IMDB Dataset.csv"))
df = pd.read_csv('IMDB Dataset.csv', encoding='utf-8')
# df.head(5)

True


## Get labels

In [6]:
_, class_names = pd.factorize(df['sentiment'])
print("Class name type: ", type(class_names))
print(class_names.to_list())

Class name type:  <class 'pandas.core.indexes.base.Index'>
['positive', 'negative']


## Prepare train and test data

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.5, random_state=1)

print("Train size: ", X_train.shape)
print("Test size: ", X_test.shape)

Train size:  (25000,)
Test size:  (25000,)


## Load pre-trained model

In [8]:
pretrain_model = 'distilbert-base-uncased'
transformer = text.Transformer(
                                pretrain_model,
                                maxlen=400,  # Max sequence(words) length
                                class_names=class_names.to_list()
                              )


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Bidirectional Encoder Representation from Transformer (BERT)

In [9]:
train = transformer.preprocess_train(X_train.to_list(), y_train.to_list())
test = transformer.preprocess_test(X_test.to_list(), y_test.to_list())


preprocessing train...
language: en
train sequence lengths:
	mean : 232
	95percentile : 593
	99percentile : 904


/usr/local/lib/python3.11/dist-packages/ktrain/utils.py:744: UserWarning: class_names argument was ignored, as they were extracted from string labels in dataset
  warnings.warn(


Is Multi-Label? False
preprocessing test...
language: en
test sequence lengths:
	mean : 230
	95percentile : 586
	99percentile : 908


In [10]:
print("Train type: ", type(train))
print("Test type: ", type(test))
print('-'*50)
print("Train shape: ", train.x.shape)
print("Test shape: ", test.x.shape)

Train type:  <class 'ktrain.text.dataset.TransformerDataset'>
Test type:  <class 'ktrain.text.dataset.TransformerDataset'>
--------------------------------------------------
Train shape:  (25000, 3, 400)
Test shape:  (25000, 3, 400)


In [11]:
print('Sample train.x')
print(train.x)

Sample train.x
[[[  101  1045  2572 ...     0     0     0]
  [    1     1     1 ...     0     0     0]
  [    0     0     0 ...     0     0     0]]

 [[  101  2023  3185 ...     0     0     0]
  [    1     1     1 ...     0     0     0]
  [    0     0     0 ...     0     0     0]]

 [[  101 16215  2890 ...     0     0     0]
  [    1     1     1 ...     0     0     0]
  [    0     0     0 ...     0     0     0]]

 ...

 [[  101  2019  5976 ...     0     0     0]
  [    1     1     1 ...     0     0     0]
  [    0     0     0 ...     0     0     0]]

 [[  101  3937  3252 ...  3246  2035   102]
  [    1     1     1 ...     1     1     1]
  [    0     0     0 ...     0     0     0]]

 [[  101  2275  1999 ...     0     0     0]
  [    1     1     1 ...     0     0     0]
  [    0     0     0 ...     0     0     0]]]


## Create model

In [12]:
model = transformer.get_classifier()
model.summary()

Model: "tf_distil_bert_for_sequence_classification_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
 dropout_39 (Dropout)        multiple                  0 (unused)
                                                                 
Total params: 66955010 (255.41 MB)
Trainable params: 66955010 (255.41 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## Train model

### Define training

In [13]:
learner = ktrain.get_learner(
    model,
    train_data=train,
    val_data=test,
    batch_size=16)

### Find the best learning rate

#### find learning rate

In [26]:
learner.lr_find() # calcurate for plotting learning rate

simulating training for different learning rates... this may take a few moments...
Epoch 1/1024


#### plotting learning

In [ ]:
learner.lr_plot()

### traning model

In [2]:
!pip install numpy==1.26.4

In [14]:
learner.fit_onecycle(lr=2e-4, epochs=1)



begin training using onecycle policy with max lr of 0.0002...
1563/1563 [==============================] - 1503s 947ms/step - loss: 0.3234 - accuracy: 0.8614 - val_loss: 0.2384 - val_accuracy: 0.9034


## Validation

In [15]:
learner.validate(class_names=transformer.get_classes)

782/782 [==============================] - 382s 486ms/step


InvalidParameterError: The 'target_names' parameter of classification_report must be an array-like or None. Got <bound method TextPreprocessor.get_classes of <ktrain.text.preprocessor.Transformer object at 0x79694105bf90>> instead.

## Review top loss

In [21]:
learner.view_top_losses(n=3, preproc=transformer)

782/782 [==============================] - 381s 487ms/step
----------
id:9948 | loss:5.11 | true:positive | pred:negative)

----------
id:15282 | loss:5.09 | true:positive | pred:negative)

----------
id:12359 | loss:5.08 | true:negative | pred:positive)



### review the data that be max loss

In [22]:
worst_index = 3629

In [23]:
X_test[worst_index]

"I don't think I need to tell you the story. For it has been told for years and years. So I will just share my feelings. I first saw Cinderella was when I was five years old. From then on I was a Disney child in a good way. The animation now seems childish and old fashioned, but that is part of its charm now. Now, in the age of High School Musical and computer generated images, it seems like people have forgotten the genius and magical essence of early Disney movies. Thankfully I was born before that so I was introduced to this classic. And it seems no matter how old I get, I turn back into that five year old watching it on VHS. Which is the true magic of Disney."

### data with tokenizing

In [24]:
test.x[worst_index]

array([[  101,  2310, 25032, ...,     0,     0,     0],
       [    1,     1,     1, ...,     0,     0,     0],
       [    0,     0,     0, ...,     0,     0,     0]])

## Prediction

### define predictor

In [25]:
predictor = ktrain.get_predictor(learner.model, transformer)

### define comment

In [26]:
com1 = "This film about political, you may fall sleep"
com2 = "This film about political, people like it. but i give one star"
com3 = "most part are bad, little part are good"
com4 = "This film about political, you should watch it"
reviews = [com1, com2, com3, com4]
print(reviews)

['This film about political, you may fall sleep', 'This film about political, people like it. but i give one star', 'most part are bad, little part are good', 'This film about political, you should watch it']


### predict

In [27]:
result = predictor.predict(reviews)
print(result)

['positive', 'positive', 'negative', 'positive']


In [28]:
predictor.predict(X_test[worst_index])

'positive'

## Save model

In [18]:
predictor.save('my_predictor')

## Load model

In [ ]:
reload_predictor = ktrain.load_predictor('my_predictor')

In [20]:
import shutil
from google.colab import files

# Replace 'your_folder' with the path to your folder
# shutil.make_archive('your_folder', 'zip', 'your_folder')
shutil.make_archive('my_predictor', 'zip', 'my_predictor')

# Download the zipped folder
files.download('my_predictor.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>